# GPT-2 + LoRA clause classification

This executable Colab notebook trains only on `data/classifier_train.csv`. The test CSV is loaded only to validate its schema and label set, and is never passed to training or model selection. Actual held-out predictions and metrics are produced only after training.

In Colab select **Runtime → Change runtime type → GPU**. You can clone your repository or upload the clean `Legal-Drafting-Assistant-Final.zip`. Training generates real weights; this project does not ship fabricated adapter files or results.

In [ ]:
%pip -q install "torch>=2.1,<3" "transformers>=4.40,<5" "peft>=0.10,<1" scikit-learn

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No CUDA GPU found. In Colab choose Runtime > Change runtime type > GPU, "
        "then rerun this cell."
    )

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile
from google.colab import drive, files

MOUNT_DRIVE = True
PROJECT_GIT_URL = ""  # Optional HTTPS URL; leave blank to upload the project ZIP.
PROJECT_DIR = Path("/content/legal-drafting-assistant")
UPLOAD_DIR = Path("/content/legal-drafting-assistant-upload")

if MOUNT_DRIVE:
    drive.mount("/content/drive")

if PROJECT_GIT_URL:
    if PROJECT_DIR.exists():
        raise FileExistsError(f"Project path already exists: {PROJECT_DIR}")
    !git clone "$PROJECT_GIT_URL" "$PROJECT_DIR"
else:
    if UPLOAD_DIR.exists():
        raise FileExistsError(f"Upload path already exists: {UPLOAD_DIR}")
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one clean project ZIP archive.")
    with zipfile.ZipFile(archives[0]) as project_archive:
        project_archive.extractall(UPLOAD_DIR)
    requirements_files = list(UPLOAD_DIR.rglob("requirements-ml.txt"))
    if len(requirements_files) != 1:
        raise FileNotFoundError("Could not locate requirements-ml.txt in the upload.")
    PROJECT_DIR = requirements_files[0].parent

os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)

In [ ]:
!pip -q install -r requirements.txt -r requirements-ml.txt

In [ ]:
import csv
from pathlib import Path
from app.clause_classifier import LEGAL_CLAUSE_LABELS
from scripts.train_gpt2_adapter import load_training_rows

TRAIN_CSV = Path("data/classifier_train.csv")
TEST_CSV = Path("data/classifier_test.csv")
train_rows = load_training_rows()

with TEST_CSV.open(encoding="utf-8-sig", newline="") as test_file:
    test_reader = csv.DictReader(test_file)
    if test_reader.fieldnames is None or set(test_reader.fieldnames) != {"text", "label"} or len(test_reader.fieldnames) != 2:
        raise ValueError("Held-out CSV must contain text and label columns.")
    test_rows = list(test_reader)

if not test_rows or any(not isinstance(row.get("text"), str) or not row["text"].strip() or not isinstance(row.get("label"), str) or not row["label"].strip() for row in test_rows):
    raise ValueError("Held-out CSV contains empty or missing fields.")
if {row["label"].strip() for row in test_rows} != set(LEGAL_CLAUSE_LABELS):
    raise ValueError("Held-out labels do not match the fixed eight-class map.")
print(f"Validated {len(train_rows)} training rows and {len(test_rows)} held-out rows.")
print("Only train_rows will be passed to the optimizer.")

In [ ]:
from scripts.train_gpt2_adapter import (
    load_base_model_and_tokenizer,
    set_random_seed,
)

set_random_seed()
tokenizer, model = load_base_model_and_tokenizer()
print("Loaded GPT-2 tokenizer and AutoModelForSequenceClassification.")

In [ ]:
from peft import get_peft_model
from scripts.train_gpt2_adapter import build_lora_config

lora_config = build_lora_config()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA target modules:", lora_config.target_modules)
print("Saved task head:", lora_config.modules_to_save)

In [ ]:
from scripts.train_gpt2_adapter import train_model

training_summary = train_model(model, tokenizer, train_rows)
training_summary

In [ ]:
from scripts.train_gpt2_adapter import OUT, save_training_artifacts

adapter_dir = save_training_artifacts(
    model, tokenizer, train_rows, training_summary, output_dir=OUT, training_csv=TRAIN_CSV
)
required_files = (
    "adapter_config.json",
    "config.json",
    "label_mapping.json",
    "training_config.json",
    "training_metadata.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
)
missing_files = [name for name in required_files if not (adapter_dir / name).is_file()]
weight_files = [name for name in ("adapter_model.safetensors", "adapter_model.bin") if (adapter_dir / name).is_file()]
if missing_files or not weight_files:
    missing_artifacts = missing_files + ([] if weight_files else ["adapter_model.safetensors or adapter_model.bin"])
    raise FileNotFoundError(f"Adapter save is incomplete: {missing_artifacts}")
print("Saved model, tokenizer, label map, training config and metadata to", adapter_dir)

In [ ]:
adapter_archive = shutil.make_archive("/content/legal_clause_adapter", "zip", adapter_dir)
files.download(adapter_archive)

In [ ]:
from app.clause_classifier import LegalClauseClassifier

classifier = LegalClauseClassifier(adapter_dir)
classifier._load()
print("Reloaded backend:", classifier.name)

In [ ]:
sample_clause = "The supplier shall indemnify the customer against third-party claims."
sample_prediction = classifier.predict(sample_clause)
print(sample_prediction)
assert sample_prediction["predicted_category"] in LEGAL_CLAUSE_LABELS
assert 0.0 <= sample_prediction["confidence"] <= 1.0
assert set(sample_prediction["probabilities"]) == set(LEGAL_CLAUSE_LABELS)

In [ ]:
from app.core import evaluate_all_models

evaluation_report = evaluate_all_models(test_csv=TEST_CSV)
print("Baseline accuracy:", evaluation_report["classifiers"]["tfidf_logreg_baseline"]["accuracy"])
print("GPT-2 status:", evaluation_report["classifiers"]["gpt2_lora"]["status"])
assert evaluation_report["classifiers"]["gpt2_lora"]["status"] == "evaluated"

In [ ]:
from app.evaluation import save_evaluation_report

report_path = save_evaluation_report(evaluation_report, Path("outputs/evaluation_report.json"))
print("Saved actual baseline and GPT-2 held-out results to:", report_path)
print("Adapter artifacts are available at:", adapter_dir)